<a href="https://colab.research.google.com/github/naritsaraLnWZA007/SC612104/blob/main/%E0%B8%AA%E0%B8%B3%E0%B9%80%E0%B8%99%E0%B8%B2%E0%B8%82%E0%B8%AD%E0%B8%87_SC612104_Session9_Pandas_Advanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SC 612 104 Essential Data Science
## คาบที่ 9: groupby, agg, sort_values, value_counts, pivot_table, merge/join, concat, apply + lambda

**ผู้สอน:** อ.ดร.พิชญา วิรัชโชติเสถียร (Pitchaya Wiratchotisatian)
**ภาควิชาสถิติ คณะวิทยาศาสตร์ มหาวิทยาลัยขอนแก่น**

---

### เนื้อหาในคาบนี้

**ส่วนที่ 1: การสรุปและเรียงข้อมูล**
1. `groupby()` — จัดกลุ่มข้อมูล
2. `agg()` — สรุปสถิติหลายแบบพร้อมกัน
3. `sort_values()` — เรียงลำดับข้อมูล
4. `value_counts()` — นับความถี่ของค่า
5. `pivot_table()` เบื้องต้น — ตารางสรุปแบบ 2 มิติ

**ส่วนที่ 2: การรวมข้อมูลจากหลายตาราง**\
6. `merge()` / `join()` — รวม DataFrame ตามคีย์\
7. `concat()` — ต่อ DataFrame เข้าด้วยกัน

**ส่วนที่ 3: การประมวลผลแบบกำหนดเอง**\
8. `apply()` + `lambda` — เขียนฟังก์ชันกำหนดเองไปใช้กับข้อมูล

**แบบฝึกหัดท้ายคาบ**

> 📌 **ก่อนเริ่ม:** Notebook นี้ต่อเนื่องจาก `SC612104_Session8_Pandas_Basics.ipynb` (Series, DataFrame, read_csv, info, describe, loc/iloc, boolean indexing) — ถ้ายังไม่คล่องคาบที่แล้ว แนะนำให้กลับไปทวนก่อน

> 📁 **ไฟล์ที่ต้องใช้ในคาบนี้:**
> - `world_weather_sample.csv` — ข้อมูลอากาศจำลอง (ไฟล์เดิมจาก Session 8)
> - `city_info.csv` — ข้อมูลเมือง (ประชากร, timezone) **ไฟล์สะอาด ไม่มี missing values หรือ outliers** — ใช้สอน merge/join
> - `region_info.csv` — ข้อมูลภูมิภาค (continent code, GDP growth) **ไฟล์สะอาดเช่นกัน** — ใช้สอน merge หลายตาราง
>
> อัปโหลดทั้ง 3 ไฟล์เข้า Colab ก่อนเริ่ม (ลากไฟล์ลงในแถบ Files ทางซ้าย หรือใช้โค้ดอัปโหลดด้านล่าง)


In [ ]:
# # ถ้าใช้ Google Colab และยังไม่ได้อัปโหลดไฟล์ ให้รันเซลล์นี้เพื่ออัปโหลดทั้ง 3 ไฟล์พร้อมกัน
# # (เลือกได้หลายไฟล์ตอนที่หน้าต่างอัปโหลดเปิดขึ้นมา)

# try:
#     from google.colab import files
#     uploaded = files.upload()   # เลือก world_weather_sample.csv, city_info.csv, region_info.csv
# except ImportError:
#     print("ไม่ได้รันบน Colab - ข้ามขั้นตอนนี้ (ตรวจสอบว่าไฟล์ CSV ทั้ง 3 อยู่ในโฟลเดอร์เดียวกับ Notebook นี้)")

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("https://raw.github.com/PitchayaW/SC-612-104-Essential-Data-Science/refs/heads/main/Module-A-Notebooks/Dataset/world_weather_sample.csv")
df["date"] = pd.to_datetime(df["date"])   # แปลงเป็น datetime เหมือนที่เรียนมาจาก Session 8
print("โหลดข้อมูลสำเร็จ! ขนาด:", df.shape)
df.head()

โหลดข้อมูลสำเร็จ! ขนาด: (2075, 10)


,record_id,date,city,country,region,temperature_c,humidity_pct,rainfall_mm,wind_speed_kmh,pressure_hpa
0,983,2025-03-24,New York,USA,North America,15.7,72.6,6.3,15.9,1005.4
1,220,2025-02-09,Tokyo,Japan,Asia,18.3,60.6,0.0,20.5,1015.1
2,1022,2025-02-01,Los Angeles,USA,North America,20.6,46.8,1.2,8.6,1013.5
3,469,2025-01-19,Mumbai,India,Asia,31.0,73.4,8.0,11.8,1000.1
4,1834,2025-02-03,Cape Town,South Africa,Africa,19.1,55.0,0.0,16.6,1015.1


---
## 1. `groupby()` — จัดกลุ่มข้อมูล

นี่คือคำสั่งที่ทรงพลังที่สุดอันหนึ่งของ pandas — ใช้ตอบคำถามแบบ **"แยกตาม..."** เช่น *"อุณหภูมิเฉลี่ยแยกตามเมืองเป็นเท่าไหร่?"*

**แนวคิดหลักของ `groupby()` มี 3 ขั้นตอน (เรียกว่า Split-Apply-Combine):**
1. **Split** — แบ่งข้อมูลเป็นกลุ่มๆ ตามคอลัมน์ที่กำหนด
2. **Apply** — ทำการคำนวณบางอย่างกับแต่ละกลุ่ม (เช่น หาค่าเฉลี่ย)
3. **Combine** — รวมผลลัพธ์ของทุกกลุ่มกลับมาเป็นตารางเดียว

เปรียบเทียบกับสิ่งที่เรียนมา: คล้ายกับการใช้ `for` loop วน list ของ object แล้วจัดกลุ่มด้วย dict — แต่ `groupby()` ทำทั้งหมดให้อัตโนมัติในคำสั่งเดียว และเร็วกว่ามาก


In [ ]:
# อุณหภูมิเฉลี่ยแยกตามเมือง
city_avg_temp = df.groupby("city")["temperature_c"].mean()
print(city_avg_temp)
print(type(city_avg_temp))   # ได้ Series กลับมา (index คือชื่อเมือง)

city
Auckland        17.804444
Bangkok         32.453846
Beijing         15.598876
Berlin          12.383146
Buenos Aires    19.171910
Cairo           26.089888
Cape Town       30.162921
Chiang Mai      29.680220
Lagos           29.494253
Lima            21.490000
London          13.490909
Los Angeles     20.657303
Mexico City     19.187778
Moscow           7.233333
Mumbai          30.872222
Nairobi         21.440000
New York        15.476667
Paris           25.811236
Sao Paulo       22.841573
Singapore       40.083333
Sydney          20.347778
Tokyo           18.414773
Toronto         11.613636
Name: temperature_c, dtype: float64
<class 'pandas.core.series.Series'>


### 1.1 `groupby()` กับหลายคอลัมน์พร้อมกัน

จัดกลุ่มตามมากกว่า 1 คอลัมน์ได้ — ส่ง list ของชื่อคอลัมน์เข้าไป (เชื่อมกับ list ที่เรียนมา)


In [ ]:
# อุณหภูมิเฉลี่ย แยกตามภูมิภาคและเมือง (2 ระดับ)
region_city_avg = df.groupby(["region", "city"])["temperature_c"].mean()
print(region_city_avg.head(10))

### 1.2 `groupby()` กับหลายคอลัมน์ผลลัพธ์ (ไม่ใช่แค่คอลัมน์เดียว)

ถ้าไม่เจาะจงคอลัมน์ตอน aggregate จะได้ผลลัพธ์ของทุกคอลัมน์ตัวเลขพร้อมกัน


In [ ]:
# ค่าเฉลี่ยของทุกคอลัมน์ตัวเลข แยกตามเมือง
city_summary = df.groupby("city")[["temperature_c", "humidity_pct", "rainfall_mm"]].mean()
print(city_summary)

### 1.3 ฟังก์ชันสรุปที่ใช้บ่อยกับ `groupby()`

ใช้ฟังก์ชันสถิติที่เรียนมาจาก NumPy/pandas ต่อท้าย `groupby()` ได้เลย: `.mean()`, `.sum()`, `.count()`, `.max()`, `.min()`, `.std()`, `.median()`


In [ ]:
print("ผลรวมฝนตกแต่ละเมือง (รวม 90 วัน):")
print(df.groupby("city")["rainfall_mm"].sum().sort_values(ascending=False).head())

print("\nจำนวนแถวข้อมูลของแต่ละเมือง (เช็คว่าครบ 90 วันไหม):")
print(df.groupby("city")["temperature_c"].count().head())   # .count() ไม่นับ missing values!

ผลรวมฝนตกแต่ละเมือง (รวม 90 วัน):


NameError: name 'df' is not defined

**🔍 สังเกต:** `.count()` ในตัวอย่างข้างบนอาจไม่เท่ากับ 90 ทุกเมือง เพราะ `.count()` **ไม่นับค่าที่เป็น missing (NaN)** — ถ้าอยากนับจำนวนแถวทั้งหมดรวม missing ด้วย ให้ใช้ `.size()` แทน


In [ ]:
print("เทียบ .count() (ไม่รวม missing) กับ .size() (รวมทุกแถว):")
comparison = pd.DataFrame({
    "count": df.groupby("city")["temperature_c"].count(),
    "size": df.groupby("city").size()
})
print(comparison.head())

---
## 2. `agg()` — สรุปสถิติหลายแบบพร้อมกัน

ถ้าอยากได้สถิติ**หลายแบบพร้อมกัน** (เช่น ทั้ง mean และ max และ min ในคำสั่งเดียว) ใช้ `.agg()` แทนการเขียน `.mean()`, `.max()`, `.min()` แยกกัน


In [ ]:
# สรุปหลายสถิติพร้อมกันด้วย list ของชื่อฟังก์ชัน (เชื่อมกับ list ที่เรียนมา)
city_stats = df.groupby("city")["temperature_c"].agg(["mean", "min", "max", "std"])
print(city_stats)

### 2.1 `agg()` กับหลายคอลัมน์ คนละสถิติกัน (ใช้ dict — เชื่อมกับ dict ที่เรียนมา)

ระบุ dict ที่ key คือชื่อคอลัมน์ value คือฟังก์ชันสถิติที่ต้องการ — ยืดหยุ่นกว่าวิธีแรกมาก


In [ ]:
# คนละคอลัมน์ คนละสถิติ: อุณหภูมิเอาเฉลี่ย, ฝนเอาผลรวม, ความเร็วลมเอาค่าสูงสุด
custom_summary = df.groupby("city").agg({
    "temperature_c": "mean",
    "rainfall_mm": "sum",
    "wind_speed_kmh": "max",
})
print(custom_summary)

### 2.2 `agg()` พร้อมตั้งชื่อคอลัมน์ผลลัพธ์เอง (Named Aggregation)

วิธีที่อ่านง่ายที่สุดและนิยมใช้ในงานจริง — ตั้งชื่อคอลัมน์ผลลัพธ์ได้ตรงๆ ไม่ต้องมาเปลี่ยนชื่อทีหลัง


In [ ]:
summary = df.groupby("city").agg(
    avg_temp=("temperature_c", "mean"),
    total_rain=("rainfall_mm", "sum"),
    max_wind=("wind_speed_kmh", "max"),
    days_recorded=("temperature_c", "count"),
)
print(summary.head(10))

---
## 3. `sort_values()` — เรียงลำดับข้อมูล

ใช้เรียงแถวของ DataFrame ตามค่าในคอลัมน์ที่กำหนด — เทียบกับ `.sort()` ของ list ที่เรียนมา แต่ทำงานกับตารางทั้งตาราง


In [ ]:
# เรียงเมืองตามอุณหภูมิเฉลี่ย จากน้อยไปมาก (default)
city_avg = df.groupby("city")["temperature_c"].mean().reset_index()
city_avg_sorted = city_avg.sort_values("temperature_c")
print(city_avg_sorted.head())

print("\nเรียงจากมากไปน้อย (ascending=False):")
city_avg_sorted_desc = city_avg.sort_values("temperature_c", ascending=False)
print(city_avg_sorted_desc.head())

NameError: name 'df' is not defined

**หมายเหตุ:** `.reset_index()` ใช้แปลงผลลัพธ์จาก `groupby()` (ที่ city กลายเป็น index) กลับเป็น DataFrame ปกติที่มี city เป็นคอลัมน์ธรรมดา — ใช้บ่อยมากหลัง `groupby()` เวลาต้องการเรียงลำดับหรือใช้ `.loc[]`/`.iloc[]` ต่อ

### 3.1 เรียงตามหลายคอลัมน์พร้อมกัน


In [ ]:
# เรียงตาม region ก่อน แล้วเรียงตาม temperature_c ภายในกลุ่มเดียวกัน
sorted_multi = df.sort_values(["region", "temperature_c"], ascending=[True, False])
print(sorted_multi[["region", "city", "temperature_c"]].head(10))

### 3.2 ตัวอย่างที่ใช้บ่อย: หา Top N ด้วย `sort_values()` + `head()`


In [ ]:
# หา 5 วันที่ฝนตกหนักที่สุดในข้อมูลทั้งหมด
top_rainy_days = df.sort_values("rainfall_mm", ascending=False).head(5)
print(top_rainy_days[["city", "date", "rainfall_mm"]])

# pandas มี .nlargest() / .nsmallest() ที่ทำสิ่งเดียวกันแบบสั้นกว่า
print("\nเทียบกับ .nlargest() (ทำสิ่งเดียวกัน เขียนสั้นกว่า):")
print(df.nlargest(5, "rainfall_mm")[["city", "date", "rainfall_mm"]])

---
## 4. `value_counts()` — นับความถี่ของค่า

ใช้นับว่าแต่ละค่าที่ไม่ซ้ำกัน (unique value) ปรากฏกี่ครั้ง — เทียบกับการใช้ `set()` หา unique values แล้ววน loop นับเอง (ที่เรียนมาจาก Collections) แต่ `.value_counts()` ทำให้ในคำสั่งเดียว


In [ ]:
print(df["region"].value_counts())

**สังเกต:** ผลลัพธ์เรียงจากมากไปน้อยอัตโนมัติ — เห็นภาพรวมได้ทันทีว่าภูมิภาคไหนมีข้อมูลมาก/น้อย

### 4.1 `value_counts()` แบบเปอร์เซ็นต์


In [ ]:
print("เป็นสัดส่วน (normalize=True):")
print(df["region"].value_counts(normalize=True))   # ได้สัดส่วน 0-1 แทนจำนวนแถวจริง

print("\nคูณ 100 ให้เป็นเปอร์เซ็นต์:")
print((df["region"].value_counts(normalize=True) * 100).round(1))

### 4.2 ใช้ `value_counts()` ตรวจสอบความสะอาดของข้อมูล (เชื่อมกับ Session 8)

จากที่รู้ว่าคอลัมน์ `country` มีตัวพิมพ์ไม่สม่ำเสมอ (จาก Session 8) — `value_counts()` ช่วยให้เห็นปัญหานี้ชัดเจน


In [ ]:
print("จำนวนค่าไม่ซ้ำของ country (ตามตัวพิมพ์จริง):")
print(df["country"].value_counts())
print(f"\nจำนวน unique values ทั้งหมด: {df['country'].nunique()}")
print("(ถ้าได้มากกว่า 23 ประเทศ แสดงว่ามีตัวพิมพ์ไม่สม่ำเสมอปนอยู่ เช่น 'Thailand' กับ 'THAILAND' ถูกนับแยกกัน)")

จำนวนค่าไม่ซ้ำของ country (ตามตัวพิมพ์จริง):


NameError: name 'df' is not defined

---
## 5. `pivot_table()` เบื้องต้น — ตารางสรุปแบบ 2 มิติ

`pivot_table()` คือการสรุปข้อมูลแบบ "ตาราง 2 มิติ" — คล้าย Pivot Table ใน Excel มาก โดยกำหนด:
- **index** → ค่าที่จะอยู่เป็น "แถว"
- **columns** → ค่าที่จะอยู่เป็น "คอลัมน์"
- **values** → ค่าที่จะเอามาคำนวณ
- **aggfunc** → ฟังก์ชันสรุปที่ใช้ (default คือ `mean`)

เปรียบเทียบกับ `groupby()`: `groupby()` ให้ผลลัพธ์เป็น "รายการยาวๆ" ส่วน `pivot_table()` จัดเรียงผลลัพธ์ให้เป็น "ตาราง" ที่อ่านง่ายกว่าเมื่อมี 2 มิติที่อยากเปรียบเทียบ


In [ ]:
# สร้างคอลัมน์เดือนจาก date (ใช้ .dt accessor ที่เรียนมาจาก Session 8)
df["month"] = df["date"].dt.month

# pivot table: แถว = region, คอลัมน์ = month, ค่า = อุณหภูมิเฉลี่ย
pivot = df.pivot_table(
    index="region",
    columns="month",
    values="temperature_c",
    aggfunc="mean"
)
print(pivot.round(1))

**อ่านผลลัพธ์:** แต่ละแถวคือภูมิภาค แต่ละคอลัมน์คือเดือน (1, 2, 3 = มกราคม, กุมภาพันธ์, มีนาคม) ค่าในตารางคืออุณหภูมิเฉลี่ยของภูมิภาคนั้นในเดือนนั้น — มองเห็นแนวโน้มตามเวลาและภูมิภาคได้ในตารางเดียว ซึ่งยากกว่าถ้าใช้ `groupby()` อย่างเดียว

### 5.1 `pivot_table()` กับ `aggfunc` หลายแบบ


In [ ]:
# ดูทั้งค่าเฉลี่ยและผลรวมพร้อมกัน
pivot_multi = df.pivot_table(
    index="region",
    columns="month",
    values="rainfall_mm",
    aggfunc=["mean", "sum"]
)
print(pivot_multi.round(1))

### 5.2 `pivot_table()` กับหลายคอลัมน์ values พร้อมกัน


In [ ]:
pivot_v2 = df.pivot_table(
    index="city",
    values=["temperature_c", "humidity_pct", "rainfall_mm"],
    aggfunc="mean"
)
print(pivot_v2.round(1).head(10))

### 5.3 จัดการ missing values ใน `pivot_table()` ด้วย `fill_value`

ถ้าบางช่องในตารางไม่มีข้อมูลเลย (เช่น ภูมิภาคนั้นไม่มีข้อมูลเดือนนั้น) จะได้ `NaN` — ใส่ `fill_value` เพื่อแทนที่ด้วยค่าที่ต้องการ


In [ ]:
pivot_filled = df.pivot_table(
    index="region",
    columns="month",
    values="temperature_c",
    aggfunc="mean",
    fill_value=0   # แทน NaN ด้วย 0 (เลือกค่าที่เหมาะกับบริบทข้อมูลเสมอ ไม่ใช่ใส่ 0 ทุกครั้ง)
)
print(pivot_filled.round(1))

NameError: name 'df' is not defined

---
## 6. `merge()` / `join()` — รวม DataFrame ตามคีย์

จนถึงตอนนี้ทำงานกับ `world_weather_sample.csv` ไฟล์เดียว — แต่งานจริงมักมีข้อมูลกระจายอยู่หลายตาราง (เหมือนตารางในฐานข้อมูล) ต้อง **รวม (join)** เข้าด้วยกันก่อนวิเคราะห์

`pd.merge()` คือคำสั่งหลักที่ใช้รวม DataFrame 2 ตัวเข้าด้วยกัน โดยอ้างอิงจาก **คีย์ร่วม** (คอลัมน์ที่มีอยู่ในทั้งสองตาราง) — แนวคิดเดียวกับ `JOIN` ใน SQL

มาดูข้อมูลอีก 2 ตารางที่จะใช้ในหัวข้อนี้: `city_info.csv` (ข้อมูลเมือง สะอาด ไม่มีปัญหา) และ `region_info.csv` (ข้อมูลภูมิภาค)


In [ ]:
city_info = pd.read_csv("https://raw.githubusercontent.com/PitchayaW/SC-612-104-Essential-Data-Science/refs/heads/main/Module-A-Notebooks/Dataset/city_info.csv")
print("city_info.csv:")
print(city_info)

print()

region_info = pd.read_csv("https://raw.githubusercontent.com/PitchayaW/SC-612-104-Essential-Data-Science/refs/heads/main/Module-A-Notebooks/Dataset/region_info.csv")
print("region_info.csv:")
print(region_info)

### 6.1 `merge()` พื้นฐาน — รวมตามคอลัมน์ที่มีชื่อเดียวกัน

ถ้าทั้งสองตารางมีคอลัมน์ชื่อเดียวกัน (`city`) `merge()` จะใช้คอลัมน์นั้นเป็นคีย์โดยอัตโนมัติ


In [ ]:
# รวมข้อมูลอากาศรายวันกับข้อมูลเมือง (ประชากร, timezone) โดยใช้ city เป็นคีย์
weather_with_city_info = pd.merge(df, city_info[["city", "population_millions", "timezone"]], on="city")
print(weather_with_city_info[["city", "date", "temperature_c", "population_millions", "timezone"]].head())
print(f"\nshape ก่อน merge: {df.shape}, shape หลัง merge: {weather_with_city_info.shape}")

NameError: name 'pd' is not defined

**สังเกต:** จำนวนแถวยังเท่ากับ `df` เดิม (2075 แถว) เพราะ `city_info.csv` มี 1 แถวต่อเมือง และทุกเมืองใน `df` มีอยู่ใน `city_info` ครบ — แต่จำนวน**คอลัมน์**เพิ่มขึ้น เพราะดึงข้อมูล `population_millions` และ `timezone` เข้ามาด้วย

### 6.2 ชนิดของการ Merge: `how` parameter

| `how` | ความหมาย | เปรียบเทียบ SQL |
|---|---|---|
| `"inner"` (default) | เอาเฉพาะแถวที่มีคีย์ตรงกัน**ทั้งสองตาราง** | INNER JOIN |
| `"left"` | เอาทุกแถวจากตารางซ้าย แม้ไม่มีคู่ตรงกันในตารางขวา (เติม NaN) | LEFT JOIN |
| `"right"` | เอาทุกแถวจากตารางขวา | RIGHT JOIN |
| `"outer"` | เอาทุกแถวจากทั้งสองตาราง | FULL OUTER JOIN |


In [ ]:
# ตัวอย่าง: สมมติมีเมืองในข้อมูลอากาศที่ไม่มีอยู่ใน city_info (ทดสอบ how="left")
extra_city = pd.DataFrame([{"city": "Unknown City", "date": "2025-01-01", "country": "?",
                             "region": "?", "temperature_c": 25.0, "humidity_pct": 60.0,
                             "rainfall_mm": 2.0, "wind_speed_kmh": 10.0, "pressure_hpa": 1013.0}])
df_test = pd.concat([df.head(3), extra_city], ignore_index=True)   # ใช้ concat() ต่อ (สอนหัวข้อถัดไป)

# left join: เก็บทุกแถวจาก df_test แม้ไม่มีข้อมูล city_info ของ "Unknown City"
left_result = pd.merge(df_test, city_info[["city", "population_millions"]], on="city", how="left")
print(left_result[["city", "population_millions"]])
print("\nสังเกต: 'Unknown City' ได้ NaN เพราะไม่มีอยู่ใน city_info -- แต่แถวไม่ได้ถูกตัดทิ้ง (เพราะใช้ how='left')")

NameError: name 'pd' is not defined

### 6.3 `merge()` เมื่อชื่อคอลัมน์คีย์ไม่เหมือนกัน

ถ้าคอลัมน์คีย์ของสองตารางชื่อไม่เหมือนกัน ใช้ `left_on` และ `right_on` ระบุแยกกัน


In [ ]:
# สมมติว่า region_info ใช้ชื่อคอลัมน์ "region" เหมือนกับ df พอดี (กรณีนี้ใช้ on="region" ได้ตรงๆ)
weather_with_region = pd.merge(df.head(5), region_info, on="region")
print(weather_with_region[["city", "region", "continent_code", "avg_gdp_growth_pct"]])

# ตัวอย่างกรณีชื่อคอลัมน์ไม่ตรงกัน (จำลองโดยเปลี่ยนชื่อคอลัมน์ใน region_info)
region_info_renamed = region_info.rename(columns={"region": "zone_name"})
weather_with_region_v2 = pd.merge(
    df.head(5), region_info_renamed,
    left_on="region", right_on="zone_name"   # ระบุชื่อคอลัมน์คีย์ของแต่ละตารางแยกกัน
)
print("\nกรณีชื่อคอลัมน์คีย์ไม่ตรงกัน (ใช้ left_on/right_on):")
print(weather_with_region_v2[["city", "region", "zone_name", "continent_code"]])

### 6.4 รวมหลายตารางต่อกันเป็นทอด (Chained Merge)

ในงานจริงมักต้อง merge มากกว่า 2 ตาราง — ทำได้โดย merge ทีละคู่ต่อกันเป็นทอดๆ


In [ ]:
# Merge 3 ตารางเข้าด้วยกัน: weather -> city_info -> region_info
full_data = (
    df
    .merge(city_info[["city", "population_millions", "timezone"]], on="city", how="left")
    .merge(region_info, on="region", how="left")
)
print(full_data[["city", "region", "population_millions", "continent_code", "avg_gdp_growth_pct"]].head())
print(f"\nshape สุดท้าย: {full_data.shape}")

### 6.5 `.join()` — ทางเลือกที่สั้นกว่าเมื่อรวมตาม index

`.join()` คล้าย `.merge()` มาก แต่ออกแบบมาให้รวมตาม **index** เป็นหลัก (ไม่ใช่คอลัมน์) — สั้นกว่าเมื่อข้อมูลทั้งสองตารางใช้ index เดียวกันอยู่แล้ว


In [ ]:
city_info_indexed = city_info.set_index("city")[["population_millions", "timezone"]]
df_indexed = df.set_index("city")

joined = df_indexed.join(city_info_indexed)
print(joined[["date", "temperature_c", "population_millions", "timezone"]].head())

---
## 7. `concat()` — ต่อ DataFrame เข้าด้วยกัน

`pd.concat()` ใช้ "ต่อ" DataFrame หลายตัวเข้าด้วยกัน — **ต่างจาก `merge()`** ที่รวมตามคีย์ `concat()` แค่ "เอามาเรียงต่อกัน" เฉยๆ (เหมือนเอา list 2 ตัวมาต่อกันด้วย `+` ที่เรียนมา)

### 7.1 `concat()` แนวตั้ง (ต่อแถว) — กรณีใช้บ่อยที่สุด

ใช้เมื่อมีข้อมูลรูปแบบเดียวกันแต่มาจากหลายไฟล์/หลายช่วงเวลา ต้องการเอามารวมเป็นตารางเดียว


In [ ]:
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/PitchayaW/SC-612-104-Essential-Data-Science/refs/heads/main/Module-A-Notebooks/Dataset/world_weather_sample.csv')
df["date"] = pd.to_datetime(df["date"])
df['month'] = df["date"].dt.month

In [ ]:
df

In [ ]:
# จำลองข้อมูล 2 ช่วงเวลาที่แยกไฟล์กัน (เช่น ข้อมูลเดือน 1 กับเดือน 2 อยู่คนละไฟล์)
batch1 = df[df["month"] == 1]
batch2 = df[df["month"] == 2]

print(f"batch1: {batch1.shape}, batch2: {batch2.shape}")

In [ ]:
batch1

In [ ]:
batch2

In [ ]:
combined = pd.concat([batch1, batch2], ignore_index=True)
# ignore_index=True สร้าง index ใหม่ให้ต่อเนื่อง
print(f"combined: {combined.shape}")
print(combined["month"].value_counts())

NameError: name 'pd' is not defined

In [ ]:
combined

**⚠️ ข้อสำคัญ:** ควรใส่ `ignore_index=True` เสมอเมื่อ concat แนวตั้ง ไม่อย่างนั้น index จะซ้ำกัน (เช่น มี index 0 ซ้ำกัน 2 รอบจากทั้งสองตาราง) ทำให้ `.loc[]` สับสนได้

### 7.2 `concat()` แนวนอน (ต่อคอลัมน์)

ใช้ `axis=1` เมื่อต้องการต่อ DataFrame ทาง "ข้าง" (เพิ่มคอลัมน์) แทนทาง "ล่าง" (เพิ่มแถว) — ต้องระมัดระวังเรื่อง index ให้ตรงกัน ไม่อย่างนั้นข้อมูลจะเรียงผิดแถว


In [ ]:
# ตัวอย่าง concat แนวนอน - ต้อง index ตรงกันก่อน
temp_data = df[["city", "temperature_c"]].head(3)
humidity_data = df[["humidity_pct"]].head(3)

In [ ]:
horizontal_concat = pd.concat([temp_data, humidity_data], axis=1)
print(horizontal_concat)

### 7.3 `concat()` vs `merge()` — สรุปเมื่อไหร่ใช้อันไหน

| | `concat()` | `merge()` |
|---|---|---|
| ใช้เมื่อ | ข้อมูลรูปแบบเดียวกัน อยากต่อกันตรงๆ | ข้อมูลคนละตาราง มีคีย์ร่วม อยากรวมตามคีย์นั้น |
| อ้างอิงจาก | ตำแหน่ง/index | คอลัมน์คีย์ (ค่าต้องตรงกันจึงรวมกัน) |
| ตัวอย่างการใช้งาน | รวมไฟล์ข้อมูลเดือน 1 + เดือน 2 | รวมข้อมูลอากาศกับข้อมูลเมือง |


---
## 8. `apply()` + `lambda` — เขียนฟังก์ชันกำหนดเองไปใช้กับข้อมูล

บางครั้งไม่มีฟังก์ชันสำเร็จรูปของ pandas ที่ตรงกับสิ่งที่อยากทำ — `.apply()` ให้นำ**ฟังก์ชันของตัวเอง**ไปใช้กับทุกแถว/ทุกค่าใน Series หรือ DataFrame ได้ เชื่อมตรงกับเรื่อง **function** (`def`) ที่เรียนมา

### 8.1 `apply()` กับฟังก์ชันที่นิยามด้วย `def`


In [ ]:
def classify_temperature(temp):
    if temp >= 35:
        return "ร้อนจัด"
    elif temp >= 25:
        return "ร้อน"
    elif temp >= 15:
        return "อบอุ่น"
    else:
        return "เย็น"

In [ ]:
df["temperature_c"].apply(classify_temperature)

In [ ]:
df["temp_category"] = df["temperature_c"].apply(classify_temperature)
print(df[["city", "temperature_c", "temp_category"]].head(10))

**สังเกต:** `classify_temperature` คือฟังก์ชันธรรมดาที่เรียนมาจาก `def` — `.apply()` เอาฟังก์ชันนี้ไปรันกับ**ทุกค่า**ในคอลัมน์ `temperature_c` แล้วเก็บผลลัพธ์เป็นคอลัมน์ใหม่ เทียบเท่ากับการเขียน `for` loop วนทุกแถวแล้วเรียกฟังก์ชันเอง แต่สั้นและเร็วกว่า

### 8.2 `lambda` — ฟังก์ชันแบบสั้นในบรรทัดเดียว

**`lambda`** คือวิธีเขียนฟังก์ชันแบบย่อ ใช้เมื่อฟังก์ชัน**ง่ายมาก**และใช้แค่ครั้งเดียว ไม่จำเป็นต้องตั้งชื่อด้วย `def`

```python
lambda parameter: expression
```

เทียบเท่ากับ:
```python
def some_name(parameter):
    return expression
```


In [ ]:
df['temp_dv_10'] = df['temperature_c']/10
df

In [ ]:
# ฟังก์ชันง่ายๆ ที่แปลงองศาเซลเซียสเป็นฟาเรนไฮต์

# วิธีปกติด้วย def
def celsius_to_fahrenheit(c):
    return c * 9/5 + 32

In [ ]:
# วิธี lambda (ใช้ครั้งเดียว ไม่ตั้งชื่อ)
df["temp_fahrenheit"] = df["temperature_c"].apply(lambda c: c * 9/5 + 32)
print(df[["city", "temperature_c", "temp_fahrenheit"]].head())

NameError: name 'df' is not defined

In [ ]:
# ตรวจสอบว่าได้ผลลัพธ์เหมือนกับใช้ def
df["temp_fahrenheit_v2"] = df["temperature_c"].apply(celsius_to_fahrenheit)
print("\nผลลัพธ์เหมือนกันไหม:", df["temp_fahrenheit"].equals(df["temp_fahrenheit_v2"]))

### 8.3 `lambda` กับเงื่อนไข (conditional expression)

`lambda` เขียน `if-else` ได้ในบรรทัดเดียว ใช้รูปแบบ `ค่าถ้าจริง if เงื่อนไข else ค่าถ้าเท็จ` (เคยเห็นรูปแบบนี้มาแล้วตอนเรียน comprehension)


In [ ]:
# ใช้ lambda กับเงื่อนไขง่ายๆ (ไม่ต้องตั้งชื่อฟังก์ชันแยกแบบ classify_temperature)
df["is_rainy"] = df["rainfall_mm"].apply(lambda x: "ฝนตก" if x > 5 else "ไม่ตก")
print(df[["city", "rainfall_mm", "is_rainy"]].head(10))

### 8.4 `apply()` กับหลายคอลัมน์พร้อมกัน (ใช้ `axis=1`)

ถ้าฟังก์ชันต้องใช้ค่าจาก**หลายคอลัมน์**พร้อมกัน ใช้ `axis=1` แล้วรับ "แถวทั้งแถว" เข้าฟังก์ชัน (เข้าถึงแต่ละคอลัมน์เหมือน dict ที่เรียนมา)


In [ ]:
def comfort_index(row):
    # ฟังก์ชันจำลองดัชนีความสบาย จากอุณหภูมิและความชื้นร่วมกัน
    if row["temperature_c"] > 30 and row["humidity_pct"] > 70:
        return "อบอ้าว"
    elif row["temperature_c"] < 15:
        return "หนาว"
    else:
        return "สบาย"

In [ ]:
df["comfort"] = df.apply(comfort_index, axis=1)   # axis=1 -> ส่งทั้งแถวเข้าฟังก์ชัน
print(df[["city", "temperature_c", "humidity_pct", "comfort"]].head(10))
print("\nสรุปจำนวนแต่ละระดับความสบาย:")
print(df["comfort"].value_counts())

### 8.5 เมื่อไหร่ใช้ `apply()`/`lambda` และเมื่อไหร่ไม่ควร

**ควรใช้เมื่อ:** ไม่มีฟังก์ชันสำเร็จรูปของ pandas ที่ตรงกับ logic ที่ต้องการ (เช่น เงื่อนไขซับซ้อนที่ผสมหลายคอลัมน์)

**ไม่ควรใช้เมื่อ:** มีฟังก์ชัน vectorized ของ pandas/NumPy ที่ทำสิ่งเดียวกันได้อยู่แล้ว — `.apply()` ทำงานทีละแถว (ช้ากว่า) ในขณะที่ฟังก์ชัน vectorized ทำงานกับทั้งคอลัมน์พร้อมกัน (เร็วกว่ามาก เหมือนที่เรียนมาจาก NumPy)

```python
# ❌ ไม่ควรทำ - ใช้ apply() กับงานที่ vectorized ทำได้อยู่แล้ว (ช้ากว่าโดยไม่จำเป็น)
df["temp_fahrenheit"] = df["temperature_c"].apply(lambda c: c * 9/5 + 32)

# ✅ ควรทำ - ใช้ vectorized operation ตรงๆ (เร็วกว่า อ่านง่ายกว่า)
df["temp_fahrenheit"] = df["temperature_c"] * 9/5 + 32
```


In [ ]:
# เปรียบเทียบผลลัพธ์ว่าเหมือนกัน แต่วิธี vectorized เร็วกว่า
df["temp_fahrenheit_vectorized"] = df["temperature_c"] * 9/5 + 32
print("ผลลัพธ์เหมือนกับ apply() ไหม:", df["temp_fahrenheit"].equals(df["temp_fahrenheit_vectorized"]))

In [ ]:
%%timeit
df["temperature_c"] * 9/5 + 32

NameError: name 'df' is not defined

In [ ]:
%%timeit
df["temperature_c"].apply(lambda c: c * 9/5 + 32)

---
## 🧪 แบบฝึกหัดท้ายคาบ

> ลองทำเองในเซลล์ที่เตรียมไว้ด้านล่างแต่ละข้อ ไม่มีเฉลยให้ — ถ้าไม่แน่ใจให้ลองรันดูผลลัพธ์ หรือถามอาจารย์/เพื่อนได้เลย ใช้ไฟล์ `world_weather_sample.csv`, `city_info.csv`, `region_info.csv`

### ข้อ 1: groupby และ agg
1. หาอุณหภูมิเฉลี่ย ความชื้นเฉลี่ย และปริมาณฝนรวม แยกตาม `region` (ใช้ `.agg()` พร้อมตั้งชื่อคอลัมน์เอง)
2. เรียงผลลัพธ์ตามอุณหภูมิเฉลี่ยจากมากไปน้อย


In [ ]:
# เขียนคำตอบข้อ 1 ที่นี่
region_summary = df.groupby('region').agg(
    avg_temperature_c=('temperature_c', 'mean'),
    avg_humidity_pct=('humidity_pct', 'mean'),
    total_rainfall_mm=('rainfall_mm', 'sum')
).sort_values(by='avg_temperature_c', ascending=False)
display(region_summary)

,avg_temperature_c,avg_humidity_pct,total_rainfall_mm
region,,,
Asia,27.920223,66.155641,3114.7
Africa,26.766479,63.613372,1410.4
South America,21.169030,71.692720,908.6
Oceania,19.076111,71.234104,683.3
North America,16.751541,60.379825,1108.9
Europe,14.712079,71.431714,1122.5


### ข้อ 2: value_counts และ sort_values
1. หาว่าเมืองไหนมีจำนวนวันที่ฝนตกหนัก (`rainfall_mm > 10`) มากที่สุด 5 อันดับแรก (Hint: กรองด้วย boolean indexing ก่อน แล้วใช้ `value_counts()` กับคอลัมน์ city)
2. ใช้ `.nlargest()` หา 5 วันที่ลมแรงที่สุดในข้อมูลทั้งหมด


In [ ]:
# เขียนคำตอบข้อ 2 ที่นี่
heavy_rain_days = df[df['rainfall_mm'] > 10]
top_5_rainy_cities = heavy_rain_days['city'].value_counts().nlargest(5)
print("5 เมืองที่มีจำนวนวันที่ฝนตกหนักที่สุด:")
display(top_5_rainy_cities)

top_5_windy_days = df.nlargest(5, 'wind_speed_kmh')[['city', 'date', 'wind_speed_kmh']]
print("\n5 วันที่ลมแรงที่สุด:")
display(top_5_windy_days)

5 เมืองที่มีจำนวนวันที่ฝนตกหนักที่สุด:


,count
city,
Singapore,44
Bangkok,23
Lagos,22
Sao Paulo,11
Buenos Aires,9



5 วันที่ลมแรงที่สุด:


,city,date,wind_speed_kmh
357,Lima,2025-01-05,33.7
1823,Chiang Mai,2025-01-06,33.5
587,Sao Paulo,2025-03-29,32.6
1735,Mexico City,2025-02-24,31.9
1974,Cape Town,2025-02-21,31.6


### ข้อ 3: pivot_table
สร้าง pivot table ที่มี:
- index = `city`
- columns = `month`
- values = `rainfall_mm`
- aggfunc = `"sum"` (ผลรวมฝนตกแต่ละเดือนของแต่ละเมือง)

จากนั้นหาว่าเมืองไหนมีฝนตกรวมทั้ง 3 เดือนมากที่สุด (Hint: ใช้ `.sum(axis=1)` กับผลลัพธ์ pivot table เพื่อรวมทุกเดือน)


In [ ]:
df["month"] = df["date"].dt.month
pivot_rainfall = df.pivot_table(
    index="city",
    columns="month",
    values="rainfall_mm",
    aggfunc="sum",
    fill_value=0 # Fill NaN with 0 for cities with no rainfall in certain months
)
display(pivot_rainfall.round(1))

# หาเมืองที่มีฝนตกรวมทั้ง 3 เดือนมากที่สุด
pivot_rainfall['total_rainfall'] = pivot_rainfall.sum(axis=1)
most_rainy_city = pivot_rainfall.nlargest(1, 'total_rainfall')

print("\nเมืองที่มีฝนตกรวมทั้ง 3 เดือนมากที่สุด:")
display(most_rainy_city)

month,1,2,3
city,,,
Auckland,106.2,95.7,150.0
Bangkok,231.6,243.9,210.1
Beijing,77.6,87.4,85.1
Berlin,98.4,69.9,100.1
Buenos Aires,93.5,121.2,90.4
Cairo,29.0,48.0,46.0
Cape Town,98.3,62.3,78.7
Chiang Mai,109.8,137.8,188.0
Lagos,201.3,218.5,233.0



เมืองที่มีฝนตกรวมทั้ง 3 เดือนมากที่สุด:


month,1,2,3,total_rainfall
city,,,,
Singapore,321.4,276.3,272.7,870.4


### ข้อ 4: merge
1. Merge `world_weather_sample.csv` กับ `city_info.csv` โดยใช้ `city` เป็นคีย์ (เลือกใช้ `how` ที่เหมาะสม)
2. จากผลลัพธ์ที่ merge แล้ว ใช้ `groupby()` หาอุณหภูมิเฉลี่ยแยกตาม `timezone`
3. Merge ผลลัพธ์ที่ได้เพิ่มกับ `region_info.csv` ด้วย (merge 3 ตารางต่อกันเป็นทอด) แล้วแสดงคอลัมน์ `city`, `region`, `continent_code`, `population_millions` ของ 10 แถวแรก


In [24]:
import pandas as pd

# Re-load city_info and region_info to ensure they are defined in this session
city_info = pd.read_csv("https://raw.githubusercontent.com/PitchayaW/SC-612-104-Essential-Data-Science/refs/heads/main/Module-A-Notebooks/Dataset/city_info.csv")
region_info = pd.read_csv("https://raw.githubusercontent.com/PitchayaW/SC-612-104-Essential-Data-Science/refs/heads/main/Module-A-Notebooks/Dataset/region_info.csv")

df_step1 = df.merge(city_info, on="city", how="left")

avg_temp_by_timezone = df_step1.groupby("timezone")["temperature_c"].mean().reset_index()
print("อุณหภูมิเฉลี่ยแยกตาม Timezone:")
display(avg_temp_by_timezone)

final_df = (
    df
    .merge(city_info, on="city", how="left")
    .merge(region_info, left_on="region_x", right_on="region", how="left") # Corrected left_on from 'region' to 'region_x'
)
print("\nแสดงคอลัมน์ city, region, continent_code, population_millions ของ 10 แถวแรกของข้อมูลที่รวมแล้ว:")
display(final_df[["city", "region_x", "continent_code", "population_millions"]].head(10)) # Display 'region_x' to avoid confusion, or rename it

อุณหภูมิเฉลี่ยแยกตาม Timezone:


,timezone,temperature_c
0,UTC+0,13.490909
1,UTC+1,22.510566
2,UTC+10,20.347778
3,UTC+12,17.804444
4,UTC+2,28.126404
5,UTC+3,14.336667
6,UTC+5:30,30.872222
7,UTC+7,31.067033
8,UTC+8,27.909497
9,UTC+9,18.414773



แสดงคอลัมน์ city, region, continent_code, population_millions ของ 10 แถวแรกของข้อมูลที่รวมแล้ว:


,city,region_x,continent_code,population_millions
0,New York,North America,NAM,8.3
1,Tokyo,Asia,AS,13.9
2,Los Angeles,North America,NAM,3.9
3,Mumbai,Asia,AS,20.4
4,Cape Town,Africa,AF,4.6
5,Auckland,Oceania,OC,1.7
6,Buenos Aires,South America,SAM,3.1
7,Mexico City,North America,NAM,9.2
8,Buenos Aires,South America,SAM,3.1
9,Los Angeles,North America,NAM,3.9


### ข้อ 5: concat
1. แบ่งข้อมูล `world_weather_sample.csv` เป็น 2 ส่วนตาม `region` (เช่น `region == "Asia"` กับ `region != "Asia"`)
2. ใช้ `concat()` ต่อทั้ง 2 ส่วนกลับเข้าด้วยกัน (ใส่ `ignore_index=True`)
3. ตรวจสอบว่าจำนวนแถวหลัง concat เท่ากับจำนวนแถวต้นฉบับหรือไม่


In [26]:
# เขียนคำตอบข้อ 5 ที่นี่
region1 = df[df["region"] == "Asia"]
region2 = df[df["region"] != "Asia"]


print(f"region1: {region1.shape}, region2: {region2.shape}")

combined = pd.concat([region1, region2], ignore_index=True)   # ignore_index=True สร้าง index ใหม่ให้ต่อเนื่อง
print(f"combined: {combined.shape}")
print(combined["region"].value_counts())

region1: (542, 11), region2: (1533, 11)
combined: (2075, 11)
region
Asia             542
Africa           362
Europe           361
North America    360
South America    270
Oceania          180
Name: count, dtype: int64


### ข้อ 6: apply + lambda
1. เขียนฟังก์ชัน (ด้วย `def`) ชื่อ `wind_category(speed)` ที่แปลงความเร็วลมเป็นหมวดหมู่: `< 10` = "ลมเบา", `10-20` = "ลมปานกลาง", `> 20` = "ลมแรง" แล้วใช้ `.apply()` สร้างคอลัมน์ใหม่
2. ใช้ `lambda` สร้างคอลัมน์ใหม่ที่บอกว่าอุณหภูมิสูงกว่า 30°C หรือไม่ (True/False)
3. ใช้ `.apply(axis=1)` เขียนฟังก์ชันที่รวมเงื่อนไขจาก `temperature_c` และ `wind_speed_kmh` เพื่อจัดหมวดหมู่ "สภาพอากาศที่ควรเลี่ยงกิจกรรมนอกบ้าน" (กำหนดเกณฑ์เองได้)


In [27]:
# เขียนคำตอบข้อ 6 ที่นี่
def wind_category(speed):
    if speed >= 10 and speed <=20:
        return "ลมปานกลาง"
    elif speed < 10:
        return "ลมเบา"
    else:
        return "ลมแรง"

In [28]:
df["windcategory"] = df["wind_speed_kmh"].apply(wind_category)
print(df[["city", "temperature_c", "wind_speed_kmh",  "windcategory"]])

             city  temperature_c  wind_speed_kmh windcategory
0        New York           15.7            15.9    ลมปานกลาง
1           Tokyo           18.3            20.5        ลมแรง
2     Los Angeles           20.6             8.6        ลมเบา
3          Mumbai           31.0            11.8    ลมปานกลาง
4       Cape Town           19.1            16.6    ลมปานกลาง
...           ...            ...             ...          ...
2070  Mexico City           17.8            10.5    ลมปานกลาง
2071      Nairobi           20.6            13.3    ลมปานกลาง
2072      Nairobi           21.7            27.2        ลมแรง
2073      Toronto            NaN             6.9        ลมเบา
2074        Paris           11.1            20.4        ลมแรง

[2075 rows x 4 columns]


In [29]:
df["temperature_high"] = df["temperature_c"].apply(lambda x: "true" if x > 30 else "false")
print(df[["city", "temperature_c", "temperature_high"]])

             city  temperature_c temperature_high
0        New York           15.7            false
1           Tokyo           18.3            false
2     Los Angeles           20.6            false
3          Mumbai           31.0             true
4       Cape Town           19.1            false
...           ...            ...              ...
2070  Mexico City           17.8            false
2071      Nairobi           20.6            false
2072      Nairobi           21.7            false
2073      Toronto            NaN            false
2074        Paris           11.1            false

[2075 rows x 3 columns]


In [30]:
def comfort_index1(row):
    if row["temperature_c"] > 30 or row["wind_speed_kmh"] > 20:
        return "สภาพอากาศที่ควรเลี่ยงกิจกรรมนอกบ้าน"
    else:
        return "จัดกิจกรรมได้"

In [31]:
df["comfort1"] = df.apply(comfort_index1, axis=1)   # axis=1 -> ส่งทั้งแถวเข้าฟังก์ชัน
print(df[["city", "temperature_c", "wind_speed_kmh", "comfort1"]])
print("\nสรุปจำนวนแต่ละระดับความสบาย:")
print(df["comfort1"].value_counts())

             city  temperature_c  wind_speed_kmh  \
0        New York           15.7            15.9   
1           Tokyo           18.3            20.5   
2     Los Angeles           20.6             8.6   
3          Mumbai           31.0            11.8   
4       Cape Town           19.1            16.6   
...           ...            ...             ...   
2070  Mexico City           17.8            10.5   
2071      Nairobi           20.6            13.3   
2072      Nairobi           21.7            27.2   
2073      Toronto            NaN             6.9   
2074        Paris           11.1            20.4   

                                 comfort1  
0                           จัดกิจกรรมได้  
1     สภาพอากาศที่ควรเลี่ยงกิจกรรมนอกบ้าน  
2                           จัดกิจกรรมได้  
3     สภาพอากาศที่ควรเลี่ยงกิจกรรมนอกบ้าน  
4                           จัดกิจกรรมได้  
...                                   ...  
2070                        จัดกิจกรรมได้  
2071                   

### ข้อ 7: ท้าทายเพิ่มเติม (Challenge) — รวมทุกอย่างเข้าด้วยกัน
จงเขียนโค้ดที่:
1. Merge ข้อมูลอากาศกับ `city_info.csv` และ `region_info.csv` (3 ตารางรวมกัน)
2. ใช้ `apply()`/`lambda` สร้างคอลัมน์ใหม่ `temp_category` (เย็น/อบอุ่น/ร้อน/ร้อนจัด ตามเกณฑ์ที่กำหนดเอง)
3. ใช้ `groupby()` กับ `["region", "temp_category"]` แล้ว `.agg()` หาจำนวนวันและอุณหภูมิเฉลี่ยของแต่ละกลุ่ม
4. ใช้ `pivot_table()` แปลงผลลัพธ์ข้อ 3 ให้เป็นตาราง: แถว = region, คอลัมน์ = temp_category, ค่า = จำนวนวัน
5. เรียงและแสดงผลลัพธ์สุดท้ายให้อ่านง่าย


In [34]:
# เขียนคำตอบข้อ 7 ที่นี่
merged_city_info = df.merge(city_info, on="city", how="left")


full_data = merged_city_info.merge(region_info, left_on="region_x", right_on="region", how="left")


full_data = full_data.drop(columns=['region_y', 'region', 'country_y'], errors='ignore')
full_data = full_data.rename(columns={'region_x': 'region', 'country_x': 'country'})

full_data["temp_category"] = full_data["temperature_c"].apply(
    lambda x: "เย็น" if x < 20
    else "อบอุ่น" if x < 30
    else "ร้อน" if x < 35
    else "ร้อนจัด"
)


summary = (
    full_data.groupby(["region", "temp_category"])
      .agg(
          จํานวนวัน=("record_id", "count"),
          อุณหภูมิเฉลี่ย=("temperature_c", "mean")
      )
      .reset_index()
)

print("สรุปข้อมูล")
print(summary)


pivot = pd.pivot_table(
    summary,
    index="region",
    columns="temp_category",
    values="จํานวนวัน",
    fill_value=0
)

pivot = pivot.reindex(
    columns=["เย็น", "อบอุ่น", "ร้อน", "ร้อนจัด"],
    fill_value=0
).sort_index()

print("\nPivot Table")
print(pivot)

สรุปข้อมูล
           region temp_category  จํานวนวัน  อุณหภูมิเฉลี่ย
0          Africa          ร้อน         47       31.404255
1          Africa       ร้อนจัด         11      276.850000
2          Africa        อบอุ่น        223       24.627803
3          Africa          เย็น         81       17.613580
4            Asia          ร้อน        208       32.126923
5            Asia       ร้อนจัด         27       76.508333
6            Asia        อบอุ่น        160       26.690000
7            Asia          เย็น        147       15.374150
8          Europe       ร้อนจัด          6      999.000000
9          Europe        อบอุ่น          4       21.225000
10         Europe          เย็น        351       11.833618
11  North America       ร้อนจัด          3             NaN
12  North America        อบอุ่น        100       22.155000
13  North America          เย็น        257       14.649027
14        Oceania        อบอุ่น         69       22.160870
15        Oceania          เย็น        111   

---
## 🔗 เชื่อม Colab กับ GitHub

เก็บ Notebook นี้ (พร้อมไฟล์ CSV ทั้ง 3) ขึ้น GitHub ต่อจากไฟล์ก่อนหน้า

### วิธีที่ 1: เซฟจาก Colab ขึ้น GitHub ตรงๆ (สำหรับ Notebook)

1. ใน Colab ไปที่เมนู **File → Save a copy in GitHub**
2. เลือก repository เดิมที่ใช้เก็บ Notebook คาบก่อนๆ (เช่น `SC612104-coursework`)
3. ตั้งชื่อไฟล์และ commit message (เช่น `"เพิ่ม notebook: groupby, merge, pivot_table"`) แล้วกด **OK**

**⚠️ ข้อควรรู้:** วิธีนี้ push แค่ตัว `.ipynb` — ไฟล์ CSV ต้องอัปโหลดขึ้น GitHub แยกต่างหาก

### วิธีที่ 2: ใช้ Git ผ่าน Terminal (push ได้ทั้ง Notebook และ CSV ในคำสั่งเดียว)

```bash
git clone https://github.com/<username>/<repo-name>.git
cd <repo-name>
git add notebook.ipynb world_weather_sample.csv city_info.csv region_info.csv
git commit -m "เพิ่ม notebook และข้อมูล: pandas groupby/merge/pivot"
git push origin main
```